# Introdução ao PyTorch

## Objetivos
Ao final deste notebook, você deve conseguir responder às seguintes perguntas:

1. O que é um tensor no PyTorch?
2. Como o autograd calcula gradientes?
3. Como definimos um modelo com `nn.Module`?
4. Como treinamos e avaliamos uma rede neural em PyTorch?

Ao longo do caminho, vamos acompanhar um fluxo completo:

**dados → modelo → loss → gradientes → atualização dos parâmetros → avaliação**.


## 1. Tensores: o objeto básico do PyTorch

Tensores são arrays multidimensionais. Eles são parecidos com arrays do NumPy, mas com uma diferença muito importante: podem ser movidos para GPU quando disponível.

In [ ]:
import torch

T = torch.tensor([[1, 2], [3, 4]])
print(T)
print("shape:", T.shape)
print("dtype:", T.dtype)

Esta célula cria um tensor 2×2.

Observe especialmente:
- `shape`: a forma do tensor;
- `dtype`: o tipo numérico armazenado;
- `torch.tensor(...)` é a forma recomendada para criar tensores a partir de dados explícitos.

In [ ]:
# dtype escolhe o tipo do dado
T_int = torch.tensor( [ [1,2], [3, 4] ], dtype=torch.int )
T_double = torch.randn( 3, 5, 7, dtype=torch.double)

In [ ]:
x = torch.randn(2, 2, dtype=torch.float64)
y = torch.randn(2, 2, dtype=torch.float64)

print("x:\n", x)
print("\ny:\n", y)

# -----------------------------
# Operações básicas
# -----------------------------
print("\nx + y:\n", x + y)
print("\nx - y:\n", x - y)

# -----------------------------
# Operações elemento a elemento
# -----------------------------
print("\nx * y:\n", x * y)
print("\nx / y:\n", x / y)

# -----------------------------
# Outras operações
# -----------------------------
print("\nx**2:\n", x**2)
print("\nsqrt(x):\n", torch.sqrt(x))

Operações aritméticas são aplicadas elemento a elemento quando isso faz sentido.


In [ ]:
v = torch.tensor([10.0, 20.0, 30.0])
M = torch.tensor([
    [0.0, 0.0, 3.0],
    [0.0, 2.0, 0.0],
    [1.0, 0.0, 0.0],
])

# -----------------------------
# Vetor e matriz
# -----------------------------
print("v (vetor):")
print(v)
print("shape:", v.shape)

print("\nM (matriz):")
print(M)
print("shape:", M.shape)

# -----------------------------
# Produto matriz-vetor
# -----------------------------
print("\nProduto matriz-vetor:")

print("M.mv(v):")
print(M.mv(v))

print("\nM @ v:")
print(M @ v)

Aqui vemos uma diferença importante entre:
- operações elemento a elemento;
- operações lineares, como produto matriz-vetor.

Em aprendizagem profunda, ambas aparecem o tempo todo.


### Operações in-place

Operações in-place modificam o próprio tensor e, em geral, terminam com underscore `_`.


In [ ]:
T = torch.empty(2, 4)

T.fill_(0.05)
print("Tensor inicial:")
print(T)

T.add_(2)
print("\nApós T.add_(2):")
print(T)

T += torch.randn(T.size())
print("\nApós T += ruído:")
print(T)

Esse estilo é útil, mas deve ser usado com cuidado, especialmente quando o tensor participa de um grafo computacional com autograd.


In [ ]:
x = torch.randn(6, 6, dtype=torch.float64)
y = torch.randn(6, 6, dtype=torch.float64)

# -----------------------------
# Tensor original
# -----------------------------
print("x:")
print(x)
print("shape:", x.shape)

# -----------------------------
# Indexação e slicing
# -----------------------------
print("\nLinha x[3, :]:")
print(x[3, :])
print("shape:", x[3, :].shape)

print("\nColuna x[:, 3]:")
print(x[:, 3])
print("shape:", x[:, 3].shape)

print("\nElemento x[0, 0]:")
print(x[0, 0].item())

In [ ]:
# -----------------------------
# Tensor original para reshape
# -----------------------------
print("\ny:")
print(y)
print("shape:", y.shape)

# -----------------------------
# Reshape
# -----------------------------
z = y.view(36)
w = y.view(-1, 9)  # 36 elementos → (4, 9)

print("\ny.view(36):")
print(z)
print("shape:", z.shape)

print("\ny.view(-1, 9):")
print(w)
print("shape:", w.shape)

## 2. Interação com NumPy

Uma das vantagens do PyTorch é a integração natural com NumPy.


In [ ]:
import numpy as np

# -----------------------------
# NumPy → PyTorch
# -----------------------------
v = np.ones(6)
print("Array NumPy:")
print(v)
print("dtype:", v.dtype)

T = torch.from_numpy(v)
print("\nTensor PyTorch (a partir do NumPy):")
print(T)
print("dtype:", T.dtype)

# -----------------------------
# Memória compartilhada
# -----------------------------
v[0] = 99
print("\nApós modificar o NumPy:")
print("v:", v)
print("T:", T)   # também muda!

# -----------------------------
# PyTorch → NumPy
# -----------------------------
T1 = torch.randn(3, 3)
print("\nTensor PyTorch:")
print(T1)

v1 = T1.numpy()
print("\nArray NumPy (a partir do tensor):")
print(v1)

# -----------------------------
# Memória compartilhada novamente
# -----------------------------
T1[0, 0] = -42
print("\nApós modificar o tensor:")
print("T1:", T1)
print("v1:", v1)   # também muda!

Sempre que possível, o tensor e o array NumPy compartilham a mesma memória subjacente.

Esse compartilhamento é eficiente, mas exige atenção: ao alterar um objeto, você pode estar alterando o outro também.


## 3. Autograd: diferenciação automática

O coração do PyTorch é o sistema de diferenciação automática.

A ideia central é a seguinte:
- definimos uma computação no passo forward;
- o PyTorch constrói o grafo computacional;
- ao chamar `backward()`, ele calcula derivadas por regra da cadeia.

Três ideias fundamentais:
1. `requires_grad=True` diz ao PyTorch para rastrear operações;
2. `loss.backward()` preenche os campos `.grad`;
3. `optimizer.step()` usa esses gradientes para atualizar os parâmetros.


In [ ]:
# Exemplo escalar simples
x = torch.tensor(1.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

y = w * x + b   # y = 2x + 3
print("grad_fn de y:", y.grad_fn)

y.backward()

print("dy/dx =", x.grad)
print("dy/dw =", w.grad)
print("dy/db =", b.grad)

Neste exemplo,
- `x`, `w` e `b` são tensores com gradientes;
- `y` depende deles;
- quando chamamos `y.backward()`, o PyTorch calcula automaticamente as derivadas de `y` em relação a cada variável.

Este é o mecanismo básico que depois será usado para treinar redes neurais inteiras.


## 4. Módulos e camadas em `torch.nn`

Redes neurais em PyTorch normalmente são construídas a partir de subclasses de `nn.Module`.

Também usamos funções de `torch.nn.functional`, que são compatíveis com autograd.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

x = torch.randn(2, 3)
print("entrada:")
print(x)

x = F.relu(x)
print()
print("após ReLU:")
print(x)

In [ ]:
f = nn.Linear(in_features=10, out_features=4)

for name, param in f.named_parameters():
    print(name, param.size())


Uma camada linear possui parâmetros treináveis:
- uma matriz de pesos;
- um vetor de bias.

Esses parâmetros são automaticamente registrados dentro do módulo.

In [ ]:
x = torch.empty(350, 10).normal_()
y = f(x)

print("shape da entrada:", x.shape)
print("shape da saída:", y.shape)

A forma da saída acompanha a regra:
- entrada: `(batch_size, in_features)`
- saída: `(batch_size, out_features)`


## 5. Definindo uma rede feedforward simples

Agora vamos definir uma rede neural com:
- camada de entrada;
- uma camada escondida;
- função de ativação ReLU;
- camada final que produz logits para classificação.


In [ ]:
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out


Importante: a camada final **não** aplica `softmax`.

Isso é proposital, porque vamos usar `nn.CrossEntropyLoss()`, que já espera **logits crus** e aplica internamente a transformação apropriada para classificação multiclasse.


## 6. Dataset e DataLoader: MNIST

Vamos usar o dataset MNIST de dígitos manuscritos.

Aqui aparecem dois objetos muito importantes:
- `Dataset`: representa a base de dados;
- `DataLoader`: organiza os dados em mini-batches e permite embaralhamento.


In [ ]:
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

input_size = 784
hidden_size = 500
num_classes = 10
num_epochs = 5
batch_size = 100
learning_rate = 1e-3

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True,
)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
)


In [ ]:
import matplotlib.pyplot as plt

# Pegar um exemplo do dataset
image, label = train_dataset[0]

print("Label:", label)
print("Shape do tensor:", image.shape)

# Mostrar a imagem
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

No MNIST, cada imagem tem forma `(1, 28, 28)`:
- `1`: número de canais;
- `28 × 28`: altura e largura.

Como nossa rede é totalmente conectada, vamos achatar cada imagem para um vetor de tamanho `784 = 28 × 28`.


## 7. Inspecionando um único mini-batch

Antes de treinar o modelo inteiro, vale muito a pena entender exatamente o que sai do `DataLoader`.


In [ ]:
images, labels = next(iter(train_loader))

print("shape original de images:", images.shape)
print("shape de labels:", labels.shape)
print("dtype de images:", images.dtype)
print("dtype de labels:", labels.dtype)
print("primeiros labels:", labels[:10])


Aqui vemos:
- `images.shape = (batch_size, 1, 28, 28)`
- `labels.shape = (batch_size,)`

Ou seja:
- temos um lote de imagens;
- cada imagem é monocromática;
- cada rótulo é um inteiro entre `0` e `9`.


In [ ]:
images_flat = images.reshape(-1, 28 * 28)

print("shape após flatten:", images_flat.shape)

A operação de flatten é essencial neste modelo:
- antes: `(batch_size, 1, 28, 28)`
- depois: `(batch_size, 784)`

Esse é um dos pontos em que alunos mais cometem erros de forma.


## 8. Um forward completo em um batch

Antes do loop de treino, vamos seguir o fluxo completo em um único mini-batch.


In [ ]:
model = NeuralNet(input_size, hidden_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

images = images.to(device)
labels = labels.to(device)
images_flat = images.reshape(-1, 28 * 28)

outputs = model(images_flat)

print("shape de outputs:", outputs.shape)
print("5 primeiros vetores de logits:")
print(outputs[:5])

A saída do modelo tem forma `(batch_size, 10)`, porque para cada imagem o modelo produz 10 logits, um para cada classe possível.


In [ ]:
loss = criterion(outputs, labels)
print("loss =", loss.item())


`CrossEntropyLoss` recebe:
- `outputs`: logits com forma `(batch_size, num_classes)`;
- `labels`: rótulos inteiros com forma `(batch_size,)`.

Não usamos vetores one-hot aqui.


In [ ]:
optimizer.zero_grad()
loss.backward()

print("shape do gradiente de fc1.weight:", model.fc1.weight.grad.shape)
print("norma do gradiente de fc1.weight:", model.fc1.weight.grad.norm().item())

Após `loss.backward()`, cada parâmetro treinável passa a ter um gradiente armazenado em `.grad`.


In [ ]:
optimizer.step()
print("Passo de otimização concluído.")


Agora o ciclo completo de treinamento já apareceu uma vez:

**forward → loss → backward → optimizer.step()**


## 9. Loop de treinamento completo

Agora vamos repetir esse processo ao longo de vários batches e épocas.


In [ ]:
model = NeuralNet(input_size, hidden_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
import matplotlib.pyplot as plt

train_loss_history = []
test_accuracy_history = []

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.reshape(-1, 28 * 28).to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return 100 * correct / total


A função de avaliação:
- coloca o modelo em modo de avaliação com `model.eval()`;
- desliga o rastreamento de gradientes com `torch.no_grad()`.

Isso economiza memória e deixa explícito que não estamos treinando nessa etapa.


In [ ]:
total_step = len(train_loader)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images = images.reshape(-1, 28 * 28).to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(
                f"Época [{epoch+1}/{num_epochs}], "
                f"Passo [{i+1}/{total_step}], "
                f"Loss: {loss.item():.4f}"
            )

    epoch_loss = running_loss / total_step
    epoch_acc = evaluate(model, test_loader, device)

    train_loss_history.append(epoch_loss)
    test_accuracy_history.append(epoch_acc)

    print(
        f"Fim da época {epoch+1}: "
        f"train_loss = {epoch_loss:.4f}, "
        f"test_accuracy = {epoch_acc:.2f}%"
    )


Agora temos duas quantidades registradas ao final de cada época:
- loss média de treino;
- acurácia no conjunto de teste.


## 10. Visualizando o histórico de treinamento


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_loss_history, marker="o")
plt.xlabel("Época")
plt.ylabel("Loss de treino")
plt.title("Evolução da loss de treino")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), test_accuracy_history, marker="o")
plt.xlabel("Época")
plt.ylabel("Acurácia no teste (%)")
plt.title("Evolução da acurácia no teste")
plt.grid(True)
plt.show()


Esses gráficos ajudam a tornar o treinamento menos “caixa-preta”:
- esperamos que a loss de treino diminua;
- esperamos que a acurácia de teste aumente, ao menos nas primeiras épocas.


## 11. Avaliação final

Vamos rodar uma avaliação final do modelo.


In [ ]:
final_acc = evaluate(model, test_loader, device)
print(f"Acurácia final no conjunto de teste: {final_acc:.2f}%")

In [ ]:
torch.save(model.state_dict(), "model.ckpt")
print("Modelo salvo em model.ckpt")

## 12. Visualizando algumas previsões

Depois de treinar, é útil olhar exemplos concretos para conectar a saída numérica do modelo com imagens reais.


In [ ]:
model.eval()

images, labels = next(iter(test_loader))
images_device = images.reshape(-1, 28 * 28).to(device)

with torch.no_grad():
    outputs = model(images_device)
    preds = outputs.argmax(dim=1).cpu()


In [ ]:
num_examples = 8

plt.figure(figsize=(12, 3))
for i in range(num_examples):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].squeeze(0), cmap="gray")
    plt.title(f"pred={preds[i].item()}\nreal={labels[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 13. Erros comuns neste tipo de notebook

Alguns erros muito frequentes:

1. **Esquecer o flatten**  
   A rede espera entradas com forma `(batch_size, 784)`, não `(batch_size, 1, 28, 28)`.

2. **Aplicar `softmax` antes de `CrossEntropyLoss`**  
   Neste caso não é necessário, porque `nn.CrossEntropyLoss()` já trata isso internamente.

3. **Esquecer `optimizer.zero_grad()`**  
   Gradientes em PyTorch são acumulados por padrão.

4. **Avaliar sem `torch.no_grad()`**  
   Isso desperdiça memória e processamento.

5. **Esquecer `model.train()` e `model.eval()`**  
   Em redes com camadas como dropout e batch normalization, isso muda o comportamento do modelo.

6. **Confundir shapes de `outputs` e `labels`**  
   Para classificação multiclasse:
   - `outputs`: `(batch_size, num_classes)`
   - `labels`: `(batch_size,)`
